> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataVisualization/blob/main/notebooks/02b_Insets_Small_Multiples_and_Export.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Insets, Small Multiples and Export

**Series:** Cartography in Python (notebook 2 of 2)
**Time:** about 30 minutes

**In this notebook you will:**
- Add a locator inset that shows where the map is
- Add a true scale bar and a north arrow
- Compare several variables with small multiples, and decide when to share a scale
- Export for print (PDF, 300 dpi) and for the web (PNG)

**You should already know:** notebook 02a.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

In [ ]:
# mapclassify matplotlib-scalebar comes installed in the SpatialTurn GIS kernel, so there is nothing to install.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts_acs.geojson").to_crs(epsg=26916)
parks = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson").to_crs(epsg=26916)
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(epsg=26916)
tracts["pct_no_vehicle"] = np.where(tracts["households"] > 0,
                                    tracts["hh_no_vehicle"] / tracts["households"] * 100, np.nan)

county = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_county.geojson").to_crs(epsg=26916)
indiana = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_counties.geojson").to_crs(epsg=26916)
tracts["poverty_rate"] = np.where(tracts["poverty_universe"] > 0,
                                  tracts["below_poverty"] / tracts["poverty_universe"] * 100, np.nan)
tracts["pct_children"] = tracts["pop_under18"] / tracts["total_pop"] * 100

## Step 1: A locator inset

Readers who do not know Tippecanoe County need to see where it is. A small **inset** map of Indiana with the
county highlighted answers that in a glance. `ax.inset_axes()` places a second set of axes inside the first,
in fractions of the main map (left, bottom, width, height).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="NaturalBreaks", k=5, cmap="YlOrRd",
            edgecolor="white", linewidth=0.4, legend=True,
            legend_kwds={"title": "Households without\na vehicle (%)", "loc": "lower left", "fmt": "{:.0f}"},
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "No households"})
ax.set_axis_off()

inset = ax.inset_axes([0.74, 0.62, 0.26, 0.38])
indiana.plot(ax=inset, color="#e0e0e0", edgecolor="white", linewidth=0.3)
county.plot(ax=inset, color="#e31a1c")
inset.set_xticks([]); inset.set_yticks([]); inset.set_xlabel(""); inset.set_ylabel("")
inset.set_title("Indiana", fontsize=8)
plt.show()

## Step 2: Scale bar and north arrow

**matplotlib-scalebar** reads the axes units and draws a correct scale bar. Tell it that one data unit is one
meter (`dx=1`), which is true because the layers are in EPSG:26916. A north arrow is only needed when north is
not obviously up, but many readers and reviewers expect one; a simple annotation is enough.

In [ ]:
from matplotlib_scalebar.scalebar import ScaleBar

def add_north_arrow(ax, x=0.06, y=0.94):
    ax.annotate("N", xy=(x, y), xytext=(x, y - 0.09), xycoords="axes fraction", ha="center", va="center",
                fontsize=12, fontweight="bold", arrowprops=dict(facecolor="black", width=4, headwidth=10))

fig, ax = plt.subplots(figsize=(8, 8))
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="NaturalBreaks", k=5, cmap="YlOrRd",
            edgecolor="white", linewidth=0.4,
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white"})
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", length_fraction=0.25, box_alpha=0.7))
add_north_arrow(ax)
ax.set_axis_off()
plt.show()

A scale bar is only true for a map in a **projected** CRS. On a map in degrees, or in Web Mercator far from the
equator, the scale changes across the map and a single bar misleads.

## Step 3: Small multiples

To compare several variables, draw **small multiples**: the same map, repeated, one variable per panel, in a grid.
The eye compares patterns across panels far better than it compares colors within one crowded map.

Here each variable has its own units and range, so each panel gets its **own** classification (quantiles, five
classes) and its own legend.

In [ ]:
variables = {"pct_no_vehicle": "Households without a vehicle (%)",
             "poverty_rate": "Below the poverty line (%)",
             "pct_children": "Under 18 (% of residents)",
             "median_income": "Median household income ($)"}

fig, axes = plt.subplots(2, 2, figsize=(11, 11))
for ax, (col, label) in zip(axes.flat, variables.items()):
    tracts.plot(ax=ax, column=col, scheme="Quantiles", k=5, cmap="PuBu", edgecolor="white", linewidth=0.3,
                legend=True, legend_kwds={"loc": "lower left", "fontsize": 7, "fmt": "{:,.0f}"},
                missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "No data"})
    ax.set_title(label, fontsize=10, loc="left")
    ax.set_axis_off()
fig.suptitle("Four views of Tippecanoe County tracts (ACS 2020-2024)", fontsize=13, fontweight="bold", x=0.05,
             ha="left")
plt.tight_layout()
plt.show()

## Step 4: When to share a scale

If panels show the **same variable** (for example the same rate in two years, or two rates in the same units that
you want to compare directly), use the **same class breaks** in every panel. Otherwise the same dark color means
20% in one panel and 60% in the other, and readers will compare colors that are not comparable.

In [ ]:
both = pd.concat([tracts["pct_no_vehicle"], tracts["poverty_rate"]]).dropna()
shared = [10, 20, 40, 60, both.max()]
print("Shared breaks:", [round(b) for b in shared])

fig, axes = plt.subplots(1, 2, figsize=(12, 6.5))
for ax, col in zip(axes, ["pct_no_vehicle", "poverty_rate"]):
    tracts.plot(ax=ax, column=col, scheme="UserDefined", classification_kwds={"bins": shared}, cmap="PuBu",
                edgecolor="white", linewidth=0.3, legend=True,
                legend_kwds={"loc": "lower left", "fontsize": 8, "fmt": "{:.0f}", "title": "%"},
                missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "No data"})
    ax.set_title(variables[col], fontsize=10, loc="left")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

With a shared scale, it is obvious that poverty rates reach much higher than car-free rates. With separate
quantile scales (Step 3) both maps look equally "dark", which hides that difference.

## Step 5: Export

| Destination | Format | Settings |
|---|---|---|
| Print, a paper, a poster | **PDF** (vector) | Text and lines stay sharp at any size; set the figure size to the printed size in inches |
| Slides, a website | **PNG** | `dpi=200` to `300`; PNG keeps sharp edges (JPEG blurs them) |
| Editing in Illustrator or Inkscape | **SVG** | Vector, every element editable |

`bbox_inches="tight"` trims empty margins. Files appear in the file browser on the left, next to this notebook, where you can
right-click and **Download** them.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7.5))                     # 7 by 7.5 inches on paper
tracts.plot(ax=ax, column="pct_no_vehicle", scheme="NaturalBreaks", k=5, cmap="YlOrRd",
            edgecolor="white", linewidth=0.4, legend=True,
            legend_kwds={"title": "Households without\na vehicle (%)", "loc": "lower left", "fmt": "{:.0f}"},
            missing_kwds={"color": "lightgray", "hatch": "///", "edgecolor": "white", "label": "No households"})
inset = ax.inset_axes([0.74, 0.62, 0.26, 0.38])
indiana.plot(ax=inset, color="#e0e0e0", edgecolor="white", linewidth=0.3)
county.plot(ax=inset, color="#e31a1c")
inset.set_xticks([]); inset.set_yticks([]); inset.set_xlabel(""); inset.set_ylabel("")
ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", length_fraction=0.25, box_alpha=0.7))
add_north_arrow(ax)
ax.set_title("Households without a car, Tippecanoe County, Indiana", loc="left", fontsize=12, fontweight="bold")
ax.set_axis_off()
fig.text(0.02, 0.01, "Data: U.S. Census Bureau, ACS 2020-2024 5-year estimates.", fontsize=7, color="dimgray")

fig.savefig("no_vehicle_map.pdf", bbox_inches="tight")
fig.savefig("no_vehicle_map.png", dpi=300, bbox_inches="tight")
plt.show()

import os
for f in ["no_vehicle_map.pdf", "no_vehicle_map.png"]:
    print(f, round(os.path.getsize(f) / 1024), "KB")

## Try it yourself

Make a **two-panel small multiple** of the share of **children** (`pct_children`) in the whole county (left) and
zoomed to the cities (right), with the **same** class breaks in both panels, one legend, a scale bar in each
panel, and a single title. Export it as a PNG at 300 dpi.

*Hint:* Compute the breaks once with `mapclassify.Quantiles(...).bins`, then pass them to both panels with `scheme="UserDefined"`.

<details>
<summary><b>Show a solution</b></summary>

```python
import mapclassify
bins = list(mapclassify.Quantiles(tracts["pct_children"].dropna(), k=5).bins)
xmin, ymin, xmax, ymax = city.total_bounds

fig, axes = plt.subplots(1, 2, figsize=(12, 6.5))
for i, ax in enumerate(axes):
    tracts.plot(ax=ax, column="pct_children", scheme="UserDefined", classification_kwds={"bins": bins},
                cmap="BuGn", edgecolor="white", linewidth=0.3, legend=(i == 0),
                legend_kwds={"loc": "lower left", "fontsize": 8, "fmt": "{:.0f}", "title": "Under 18 (%)"})
    ax.add_artist(ScaleBar(dx=1, units="m", location="lower right", length_fraction=0.25, box_alpha=0.7))
    ax.set_axis_off()
axes[1].set_xlim(xmin - 1000, xmax + 1000); axes[1].set_ylim(ymin - 1000, ymax + 1000)
axes[0].set_title("Tippecanoe County", loc="left", fontsize=10)
axes[1].set_title("Lafayette and West Lafayette", loc="left", fontsize=10)
fig.suptitle("Share of residents under 18, by census tract", x=0.05, ha="left", fontsize=13, fontweight="bold")
fig.savefig("children_two_panels.png", dpi=300, bbox_inches="tight")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- An inset locator map (`ax.inset_axes`) shows where the map is
- `matplotlib-scalebar` draws a true scale bar for maps in meters; add a simple north arrow
- Small multiples compare variables; share class breaks only when panels show comparable values
- Export PDF for print and PNG at 200 to 300 dpi for screens

**Next:** the Accessibility in Map Design episode of this module, or the Cartography in QGIS episode for the same
techniques in a print layout.